# SET Pharma — Google Colab Pipeline

This notebook runs the existing `SET_new` Python implementation. It does not duplicate model, feature-engineering, reliability or RADCSR logic.

**Research boundary:** Inventory outcomes are simulated. Stockout labels are not observed events.

Run the cells in order. The notebook checkpoints ingestion, the model pipeline and mathematical validation separately. The current `run_model_pipeline()` is one monolithic stage; an interruption inside it requires rerunning that stage.

## 00. Runtime diagnostics

Inspect the allocated runtime before copying or processing the dataset. A GPU is optional; this workload may be limited by RAM, CPU and disk rather than GPU.

In [ ]:
import os, sys, shutil, subprocess, json, time
from pathlib import Path

print("Python:", sys.version)
if sys.version_info[:3] != (3, 13, 18):
    raise RuntimeError(
        f"This Colab notebook requires Python 3.13.18; detected {sys.version.split()[0]}. "
        "Select a Python 3.13 Colab runtime and restart the session."
    )
print("Python 3.13 runtime verified.")
print("CPU cores:", os.cpu_count())
print("Disk free (GB):", round(shutil.disk_usage('/content').free / 1024**3, 2))
try:
    import psutil
    print("RAM total (GB):", round(psutil.virtual_memory().total / 1024**3, 2))
    print("RAM available (GB):", round(psutil.virtual_memory().available / 1024**3, 2))
except ImportError:
    print("psutil not installed yet")
try:
    import torch
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    print("PyTorch is not installed; GPU diagnostics skipped")

## 01. Clone the feature branch

The notebook deliberately checks out `feature/colab-pipeline-execution`, not `main`.

In [ ]:
REPO_URL = "https://github.com/Souptik-Hazra/SET_new.git"
BRANCH = "feature/colab-pipeline-execution"
DEFAULT_NOTEBOOK_COMMIT = "4fec511e61956887559d297eb861f1414d2f5be6"
NOTEBOOK_COMMIT = DEFAULT_NOTEBOOK_COMMIT  # Override only after reviewing the replacement commit.
if NOTEBOOK_COMMIT != DEFAULT_NOTEBOOK_COMMIT:
    print(f"WARNING: NOTEBOOK_COMMIT override active: {NOTEBOOK_COMMIT}")
REPO_DIR = Path("/content/SET_new")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)

# Execute the reviewed snapshot, not the moving branch head.
subprocess.run(["git", "checkout", NOTEBOOK_COMMIT], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))
print("Repository:", REPO_DIR)
print("Branch:", subprocess.check_output(["git", "branch", "--show-current"], cwd=REPO_DIR, text=True).strip())
print("Commit:", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_DIR, text=True).strip())

## 02. Install project dependencies

Use the repository's pinned dependency ranges. Installation is performed in the active Colab runtime.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "psutil"], check=True)
print("Dependencies installed. If Colab requests a runtime restart, restart and continue from section 00.")

## 03. Mount Google Drive and configure paths

Place the source CSV at the path below in Drive, or edit `DRIVE_RAW_CSV` to match its actual location. The raw file is copied to Colab's local disk for processing; it is never uploaded to Git.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RAW_CSV = Path("/content/drive/MyDrive/Dataset/combined_secondary_care_medicines.csv")
DRIVE_ARTIFACT_ROOT = Path("/content/drive/MyDrive/SET_Pharma_Colab_Artifacts")
LOCAL_RAW_CSV = REPO_DIR / "data" / "raw" / "combined_secondary_care_medicines.csv"

if not DRIVE_RAW_CSV.is_file():
    raise FileNotFoundError(f"Upload the SCMD CSV to {DRIVE_RAW_CSV}, or update DRIVE_RAW_CSV.")
LOCAL_RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
required_bytes = DRIVE_RAW_CSV.stat().st_size
free_bytes = shutil.disk_usage("/content").free
if free_bytes < required_bytes * 1.15:
    raise RuntimeError(f"Insufficient Colab disk space: need approximately {required_bytes * 1.15 / 1024**3:.2f} GB; available {free_bytes / 1024**3:.2f} GB")
if (not LOCAL_RAW_CSV.is_file()
    or LOCAL_RAW_CSV.stat().st_size != required_bytes
    or LOCAL_RAW_CSV.stat().st_mtime_ns != DRIVE_RAW_CSV.stat().st_mtime_ns):
    print("Copying raw CSV from Drive to local Colab storage...")
    shutil.copy2(DRIVE_RAW_CSV, LOCAL_RAW_CSV)
print("Raw CSV:", LOCAL_RAW_CSV, "| GB:", round(LOCAL_RAW_CSV.stat().st_size / 1024**3, 3))
print("Persistent artifacts:", DRIVE_ARTIFACT_ROOT)

## 04. Execution controls

Keep the research defaults unless a documented experiment requires a change. Set `FORCE_* = True` only when intentionally rebuilding a stage. A checkpoint is reused only when its fingerprint matches and all expected outputs exist.

In [ ]:
from src.set_forecasting.config import get_config, validate_config

CONFIG = validate_config(get_config())
RUN_INGESTION = True
RUN_MODEL_PIPELINE = True
RUN_MATHEMATICAL_VALIDATION = True

FORCE_INGESTION = False
FORCE_MODEL_PIPELINE = False
FORCE_MATHEMATICAL_VALIDATION = False

RESTORE_CHECKPOINTS = False  # Set True only when intentionally resuming a previous run.

MIN_PERIODS = CONFIG["data"]["min_periods"]
MIN_NONZERO = CONFIG["data"]["min_nonzero"]
TRAIN_FRAC = CONFIG["data"]["train_frac"]
VALIDATION_FRAC = CONFIG["data"]["validation_frac"]
TEST_FRAC = CONFIG["data"]["test_frac"]
BOOTSTRAP_RESAMPLES = CONFIG["evaluation"]["bootstrap_resamples"]
RANDOM_SEED = CONFIG["seed"]

print(json.dumps({k: v for k, v in globals().copy().items() if k in [
    "RUN_INGESTION", "RUN_MODEL_PIPELINE", "RUN_MATHEMATICAL_VALIDATION",
    "FORCE_INGESTION", "FORCE_MODEL_PIPELINE", "FORCE_MATHEMATICAL_VALIDATION",
    "MIN_PERIODS", "MIN_NONZERO", "TRAIN_FRAC", "VALIDATION_FRAC", "TEST_FRAC", "BOOTSTRAP_RESAMPLES", "RANDOM_SEED"
]}, indent=2))


## 05. Restore previous checkpoints and import the runner

Only processed Parquet files and reports are restored from Drive. The raw CSV and repository are not copied back from the artifact folder.

In [ ]:
import importlib.util

required_imports = (
    "scripts.colab_runtime",
    "src.set_forecasting.config",
    "src.set_forecasting.pipeline",
    "src.visualization.generate_figures",
)
missing_imports = [name for name in required_imports if importlib.util.find_spec(name) is None]
if missing_imports:
    raise ImportError(f"Repository modules are not importable: {missing_imports}")
print("Repository import checks passed:", ", ".join(required_imports))

from scripts.colab_runtime import (
    environment_report, sync_artifacts, run_ingestion,
    run_models, run_mathematical_validation, DISCLAIMER
)

print(json.dumps(environment_report(), indent=2))
if RESTORE_CHECKPOINTS:
    restored = sync_artifacts(DRIVE_ARTIFACT_ROOT, REPO_DIR) if DRIVE_ARTIFACT_ROOT.exists() else []
    print("Restored artifact files:", len(restored))
else:
    for relative in ("data/processed", "output"):
        target = REPO_DIR / relative
        if target.exists():
            shutil.rmtree(target)
        target.mkdir(parents=True, exist_ok=True)
    restored = []
    print("Fresh run selected: cleared local processed artifacts, reports and checkpoint manifest.")
print(DISCLAIMER)

## 06. Phase 1 — SCMD ingestion

DuckDB streams and aggregates the CSV. The monthly cohort is written to Parquet. The raw file remains outside Git. After success, processed outputs and the stage manifest are copied to Drive.

In [ ]:
if RUN_INGESTION:
    ingestion_result = run_ingestion(
        LOCAL_RAW_CSV,
        min_periods=MIN_PERIODS,
        min_nonzero=MIN_NONZERO,
        train_frac=TRAIN_FRAC,
        val_frac=VALIDATION_FRAC,
        test_frac=TEST_FRAC,
        force=FORCE_INGESTION,
    )
    print(ingestion_result)
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Ingestion disabled by configuration")

## 07. Model implementation — visible in this notebook

The active research pipeline currently trains **three quantile LightGBM estimators (Q10, Q50, Q90)** and compares them with previous-observation and seasonal-naive baselines. The implementation below is inlined from the canonical feature-engineering and forecasting modules and is wired into the downstream pipeline before execution.

**Code-review clarification:** The older `src/forecasting/xgboost_model.py`, `lightgbm_model.py` and `sarima.py` files use a different weekly/daily data interface. They are not invoked by the active SCMD monthly pipeline, so this notebook does not present them as models that are currently trained. Adding them would require a separate, methodologically reviewed experiment rather than silently changing the current pipeline.


In [ ]:
# Canonical model implementation, inlined from src/set_forecasting for notebook transparency.
# Keep this cell synchronized with the pinned repository implementation.
import numpy as np
import pandas as pd

import numpy as np
import pandas as pd
SERIES_KEYS=["ODS_CODE","VMP_SNOMED_CODE","UNIT_OF_MEASURE_IDENTIFIER"]
TARGET="MODEL_DEMAND"
LAGS=(1,2,3,6,12)
WINDOWS=(3,6,12)

def create_features(data):
    missing=set(SERIES_KEYS+["PERIOD",TARGET,"OBSERVATION_MISSING"])-set(data.columns)
    if missing: raise ValueError(f"Missing SCMD columns: {sorted(missing)}")
    if data.empty:return data.copy()
    frames=[]
    for _,g in data.groupby(SERIES_KEYS,sort=False,dropna=False):
        g=g.sort_values("PERIOD").copy();g["PERIOD"]=pd.to_datetime(g.PERIOD)
        missing_obs=g.OBSERVATION_MISSING.fillna(True).astype(bool)
        y=pd.to_numeric(g[TARGET],errors="coerce").where(~missing_obs)
        past=y.shift(1)
        for lag in LAGS:g[f"LAG_{lag}"]=y.shift(lag)
        for w in WINDOWS:
            r=past.rolling(w,min_periods=1);n=past.rolling(w,min_periods=1).count()
            mean=r.mean();std=r.std(ddof=0)
            g[f"ROLL_MEAN_{w}"]=mean;g[f"ROLL_MEDIAN_{w}"]=r.median()
            g[f"ROLL_STD_{w}"]=std;g[f"ROLL_MIN_{w}"]=r.min();g[f"ROLL_MAX_{w}"]=r.max()
            g[f"ROLL_CV_{w}"]=std/mean.replace(0,np.nan)
            g[f"ZERO_RATIO_{w}"]=past.eq(0).rolling(w,min_periods=1).sum()/n.replace(0,np.nan)
            g[f"OBSERVED_RATIO_{w}"]=(~missing_obs).shift(1).astype(float).rolling(w,min_periods=1).mean()
        positive=past.gt(0).fillna(False).to_numpy();age=np.full(len(g),np.nan);last=None
        for i,flag in enumerate(positive):
            age[i]=np.nan if last is None else i-last
            if flag:last=i
        g["MONTHS_SINCE_NONZERO"]=age
        g["PRIOR_NONZERO_MEAN_6"]=past.where(past>0).rolling(6,min_periods=1).mean()
        g["MOMENTUM_3M"]=g.ROLL_MEAN_3/g.ROLL_MEAN_6.replace(0,np.nan)
        g["SEASONAL_RATIO_12"]=g.LAG_12/g.ROLL_MEAN_12.replace(0,np.nan)
        month=g.PERIOD.dt.month
        g["MONTH"]=month;g["QUARTER"]=g.PERIOD.dt.quarter;g["YEAR"]=g.PERIOD.dt.year
        g["MONTH_SIN"]=np.sin(2*np.pi*month/12);g["MONTH_COS"]=np.cos(2*np.pi*month/12)
        g["TREND_INDEX"]=np.arange(len(g),dtype=np.int32);g["TARGET_OBSERVED"]=~missing_obs
        if {"TOTAL_INDICATIVE_COST","POSITIVE_QUANTITY"}.issubset(g.columns):
            cost=pd.to_numeric(g.TOTAL_INDICATIVE_COST,errors="coerce")
            qty=pd.to_numeric(g.POSITIVE_QUANTITY,errors="coerce")
            unit=(cost/qty.where(qty>0)).replace([np.inf,-np.inf],np.nan)
            g["LAG_UNIT_COST_1"]=unit.shift(1)
            g["ROLL_UNIT_COST_3"]=unit.shift(1).rolling(3,min_periods=1).mean()
        frames.append(g)
    return pd.concat(frames,ignore_index=True)

def leakage_check(features):
    required=set(SERIES_KEYS+["PERIOD",TARGET,"OBSERVATION_MISSING","LAG_1"])
    if required-set(features.columns):return False
    for _,g in features.groupby(SERIES_KEYS,sort=False,dropna=False):
        g=g.sort_values("PERIOD")
        expected=pd.to_numeric(g[TARGET],errors="coerce").where(~g.OBSERVATION_MISSING.fillna(True).astype(bool)).shift(1)
        if not np.allclose(g.LAG_1.to_numpy(float),expected.to_numpy(float),equal_nan=True):return False
    return True


import numpy as np
import pandas as pd

FEATURES=["LAG_1","LAG_2","LAG_3","LAG_6","LAG_12","ROLL_MEAN_3","ROLL_MEDIAN_3","ROLL_STD_3","ROLL_CV_3","ROLL_MEAN_6","ROLL_MEDIAN_6","ROLL_STD_6","ROLL_CV_6","ROLL_MIN_6","ROLL_MAX_6","ZERO_RATIO_6","OBSERVED_RATIO_6","MONTHS_SINCE_NONZERO","PRIOR_NONZERO_MEAN_6","MOMENTUM_3M","SEASONAL_RATIO_12","MONTH","QUARTER","YEAR","MONTH_SIN","MONTH_COS","TREND_INDEX","LAG_UNIT_COST_1","ROLL_UNIT_COST_3"]

def _score(g,name):
    y=g.MODEL_DEMAND.to_numpy(float);lo=g.Q10.to_numpy(float);mid=g.Q50.to_numpy(float);hi=g.Q90.to_numpy(float)
    m=np.isfinite(y)&np.isfinite(lo)&np.isfinite(mid)&np.isfinite(hi);y,lo,mid,hi=y[m],lo[m],mid[m],hi[m]
    if not len(y):return {"MODEL":name,"N":0}
    pin=lambda q,p:float(np.mean(np.maximum(q*(y-p),(q-1)*(y-p))))
    return {"MODEL":name,"N":len(y),"MAE":float(np.mean(abs(y-mid))),
      "WAPE":float(abs(y-mid).sum()/max(abs(y).sum(),1e-9)*100),
      "PICP_80":float(np.mean((y>=lo)&(y<=hi))),"MPIW_80":float(np.mean(hi-lo)),
      "PINBALL_10":pin(.1,lo),"PINBALL_50":pin(.5,mid),"PINBALL_90":pin(.9,hi)}

def train_and_predict(series,min_train_rows=100,random_state=42,winsor_quantile=.995):
    x=series.copy()
    x["MODEL_DEMAND_WINSORIZED"]=np.nan;x["OUTLIER_995_FLAG"]=False
    x["_ORIGINAL_MODEL_DEMAND"]=pd.to_numeric(x.MODEL_DEMAND,errors="coerce")
    for _,idx in x.groupby(SERIES_KEYS,sort=False,dropna=False).groups.items():
        g=x.loc[idx];train_g=g[(g.SPLIT=="train")&(~g.OBSERVATION_MISSING.astype(bool))]
        positive=pd.to_numeric(train_g.MODEL_DEMAND,errors="coerce");positive=positive[positive>0]
        threshold=float(positive.quantile(winsor_quantile)) if len(positive) else np.inf
        values=pd.to_numeric(g.MODEL_DEMAND,errors="coerce")
        x.loc[idx,"OUTLIER_995_FLAG"]=values>threshold
        x.loc[idx,"WINSOR_LIMIT_995"]=threshold
        x.loc[idx,"MODEL_DEMAND_WINSORIZED"]=np.where(values>0,np.minimum(values,threshold),values)
    x[TARGET]=x.MODEL_DEMAND_WINSORIZED
    features=create_features(x)
    features["MODEL_DEMAND"]=features["_ORIGINAL_MODEL_DEMAND"]
    features["SERIES_ID"]=features[SERIES_KEYS].astype(str).agg("|".join,axis=1)
    categorical=[]
    for col in SERIES_KEYS:
        name=col+"_CAT";features[name]=features[col].astype("category");categorical.append(name)
    cols=[c for c in FEATURES if c in features.columns]+categorical
    for col in FEATURES:
        if col in features:features[col]=pd.to_numeric(features[col],errors="coerce")
    train=features[(features.SPLIT=="train")&features.TARGET_OBSERVED].dropna(subset=["MODEL_DEMAND_WINSORIZED"])
    validation_calendar=features[features.SPLIT=="validation"].copy()
    val=validation_calendar[validation_calendar.TARGET_OBSERVED].dropna(subset=["MODEL_DEMAND_WINSORIZED"])
    test=features[features.SPLIT=="test"].copy()
    models=[];status="naive_fallback"
    try:
        import lightgbm as lgb
        from lightgbm import LGBMRegressor
        if len(train)>=min_train_rows and len(val)>=20:
            for quantile in (.1,.5,.9):
                model=LGBMRegressor(objective="quantile",alpha=quantile,n_estimators=800,
                    learning_rate=.03,num_leaves=31,min_child_samples=30,subsample=.85,
                    colsample_bytree=.85,verbosity=-1,random_state=random_state,n_jobs=-1)
                model.fit(train[cols],train["MODEL_DEMAND_WINSORIZED"],categorical_feature=categorical,
                    eval_set=[(val[cols],val["MODEL_DEMAND_WINSORIZED"])],
                    callbacks=[lgb.early_stopping(50,verbose=False)])
                models.append(model)
            status="quantile_lightgbm"
    except ImportError:
        pass
    def predict(part):
        part=part.copy()
        if part.empty:return part
        if models:
            raw=np.column_stack([np.maximum(0,m.predict(part[cols])) for m in models])
            part[["Q10","Q50","Q90"]]=np.sort(raw,axis=1)
        else:
            part["Q50"]=part.LAG_1.fillna(part.LAG_12).fillna(0).clip(lower=0)
            part["Q10"]=part.Q50*.5;part["Q90"]=part.Q50*1.5
        part["FORECAST_MODEL"]=status
        part["NAIVE_LAST"]=part.LAG_1.fillna(0).clip(lower=0)
        part["NAIVE_SEASONAL_12"]=part.LAG_12.fillna(part.NAIVE_LAST).clip(lower=0)
        return part
    validation_pred=predict(validation_calendar);test_pred=predict(test)
    all_predictions=pd.concat([validation_pred,test_pred],ignore_index=True)
    metrics=[]
    if len(test_pred):
        metrics.append(_score(test_pred,"QuantileLightGBM" if status=="quantile_lightgbm" else "NaiveFallback"))
        for col,name in [("NAIVE_LAST","PreviousObservationNaive"),("NAIVE_SEASONAL_12","SeasonalNaive12")]:
            z=test_pred.copy();z["Q10"]=z[col]*.5;z["Q50"]=z[col];z["Q90"]=z[col]*1.5
            metrics.append(_score(z,name))
    report={"model":status,"train_rows":len(train),"validation_rows":len(val), "validation_calendar_rows":len(validation_calendar),
      "test_rows":len(test_pred),"test_observed_rows":int(test_pred.TARGET_OBSERVED.sum()),
      "validation_predictions":len(validation_pred)}
    return features,all_predictions,pd.DataFrame(metrics),report


# The canonical pipeline imports this function; patch its reference so the
# training implementation executed from this notebook is the one used below.
import src.set_forecasting.pipeline as _pipeline_module
_pipeline_module.train_and_predict = train_and_predict
print('Inline feature engineering and quantile LightGBM training code loaded.')

## 08. Forecasting, reliability, RADCSR and simulation

The downstream pipeline calls the notebook-defined `train_and_predict()` implementation above, then runs reliability estimation, RADCSR and inventory simulation. It remains a single long-running stage. If Colab disconnects while this function is running, rerun this cell; the completed ingestion checkpoint will be reused. Model-stage internal checkpoints are a future refactor, not simulated here.

In [ ]:
if RUN_MODEL_PIPELINE:
    model_result = run_models(
        LOCAL_RAW_CSV,
        min_periods=MIN_PERIODS,
        min_nonzero=MIN_NONZERO,
        train_frac=TRAIN_FRAC,
        val_frac=VALIDATION_FRAC,
        test_frac=TEST_FRAC,
        bootstrap=BOOTSTRAP_RESAMPLES,
        seed=RANDOM_SEED,
        force=FORCE_MODEL_PIPELINE,
    )
    print(model_result)
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Model pipeline disabled by configuration")

## 09. Phase B mathematical validation

This requires the forecast and simulation artifacts from the model stage. It evaluates the current implementation and writes JSON and Markdown reports.

In [ ]:
if RUN_MATHEMATICAL_VALIDATION:
    math_result = run_mathematical_validation(
        bootstrap=BOOTSTRAP_RESAMPLES,
        seed=RANDOM_SEED,
        force=FORCE_MATHEMATICAL_VALIDATION,
    )
    print("Mathematical validation status:", math_result["status"])
    saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
    print("Persisted files to Drive:", len(saved))
else:
    print("Mathematical validation disabled by configuration")

## 10. Generate, save and display research figures

Figures are generated from the pipeline's saved Parquet and CSV artifacts. The generator does not insert illustrative metric values. Figures based on synthetic inventory outcomes are explicitly labelled as simulated. Each PNG is saved under `output/figures/`, displayed below, and synchronized to Google Drive.

In [ ]:
from IPython.display import Image, display
from src.visualization.generate_figures import generate_figures

generated_figures = generate_figures(REPO_DIR)
if generated_figures:
    for figure_path in generated_figures:
        print(figure_path.name)
        display(Image(filename=str(figure_path), width=1000))
else:
    print("No figures were generated. Inspect output/figures/figure_generation_report.json for missing inputs.")

saved = sync_artifacts(REPO_DIR, DRIVE_ARTIFACT_ROOT)
print("Persisted figures and reports to Drive:", len(saved))
print(DISCLAIMER)

## 11. Review generated outputs

Inspect reports before drawing research conclusions. Simulated stockout labels are not observed hospital stockout events.

In [ ]:
report_paths = [
    REPO_DIR / "output" / "pipeline_report.json",
    REPO_DIR / "output" / "mathematical_validation.json",
    REPO_DIR / "output" / "final_verdict.md",
    REPO_DIR / "output" / "figures" / "figure_generation_report.json",
]
for path in report_paths:
    print(("FOUND  " if path.is_file() else "MISSING ") + str(path))
    if path.is_file(): print("  Size:", path.stat().st_size, "bytes")

processed = REPO_DIR / "data" / "processed"
print("\nProcessed artifacts:")
for path in sorted(processed.glob("*")):
    if path.is_file(): print(f"{path.name:45s} {path.stat().st_size / 1024**2:8.2f} MB")

figures_dir = REPO_DIR / "output" / "figures"
print("\nSaved figures:")
for path in sorted(figures_dir.glob("*.png")):
    print(f"{path.name:48s} {path.stat().st_size / 1024:.1f} KB")

print("\n", DISCLAIMER)

## 12. Re-running selected stages

- To rerun only mathematical validation, set `RUN_INGESTION = False`, `RUN_MODEL_PIPELINE = False`, `RUN_MATHEMATICAL_VALIDATION = True` and rerun the relevant cells.
- To rebuild forecasts and downstream outputs from an existing monthly series, disable ingestion and force the model stage.
- To rebuild the monthly cohort, force ingestion; downstream stages should then be rerun.
- If the runtime disconnects, reconnect, rerun setup and Drive restore cells, then rerun the desired stage cells.

The checkpoint fingerprint includes source-file metadata, hashes of the code files used by that stage, the configuration hash and stage parameters. The Git revision is recorded in the manifest for audit. Changing relevant implementation code or configuration invalidates the affected checkpoint.